# Motor Condition Classification with an Artificial Neural Network
**Research project:** *A Machine Learning Approach to Proactive Machine Failure Prediction for Modern Manufacturing Systems*

**Notebook version 1.0.0** &nbsp;|&nbsp; Runs on Google Colab (CPU is enough) &nbsp;|&nbsp; Option A: condition classification

> **How to use this notebook**
> 1. Upload `motor_predictive_maintenance_master.csv` to Colab (left folder icon → upload), or mount Google Drive.
> 2. Fill in the **label provenance** form in Section 2 (this is the most important cell for your viva).
> 3. Choose **Runtime → Run all**. Every number in this notebook is computed when you run it. Nothing is typed in by hand.
> 4. Download the zipped artifact bundle from the last cell.
>
> **Do not edit and re-run the test-set cell (Section 16) to "improve" results.** That would turn the test set into a second validation set and invalidate your claims.

## 1. Research objective and experimental scope

### What this notebook answers (Option A: condition classification)
> *Can an Artificial Neural Network classify the **condition** of an induction motor (Normal / Alert / Danger) from a single set of vibration, gE, temperature and operating-condition readings, **for motors it has never seen during training**?*

### What this notebook does NOT answer (Option B: future failure prediction)
Predicting that a motor **will fail within a future time window** needs two things that the current file does not contain:
1. **Verified failure event timestamps** (when a motor actually failed or was repaired because of a fault).
2. A **prediction horizon** (for example "failure within the next 14 days") defined *before* modelling.

`Danger` is a **condition state**, not a recorded failure. Treating it as a failure would be a claim the data cannot support. Section 18b lists the exact evidence you would need to collect for Option B.

### Design summary
| Decision | Choice | Why (short) |
|---|---|---|
| Target | `condition_label` (3 classes, ordered) | Only target in the file; **provisional** until provenance is documented |
| Unit of generalisation | **Unseen motors** | A deployed model will meet motors it was not trained on |
| Split | **Motor-grouped** train / validation / test | Repeated readings of one motor are near-duplicates; row-level splitting would inflate scores |
| Preprocessing | Standardisation **fitted on training motors only** | Prevents information from validation/test leaking into training |
| Main metric | **Macro-F1** (+ per-class recall) | Classes are imbalanced; accuracy would reward ignoring `Danger` |
| Tuning | Validation set only | Test set stays untouched until one final evaluation |
| Models | ANN (primary, declared in advance) vs Dummy, Logistic Regression, Random Forest | Baselines show whether the ANN adds anything |

### Honest limits
* Results describe **this dataset only**. They are not evidence of real-world failure-prediction ability.
* If the labels were produced by a rule applied to the same sensor columns, a model can learn the rule rather than motor health (checked in Section 12b).

## 2. Imports, random seeds, reproducibility settings and the label-provenance gate

In [ ]:
import os
# These must be set BEFORE TensorFlow is imported.
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import sys, json, hashlib, random, platform, warnings, datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, ParameterSampler, ParameterGrid
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, precision_recall_fscore_support,
                             confusion_matrix, matthews_corrcoef, cohen_kappa_score, log_loss, roc_auc_score,
                             average_precision_score, recall_score, precision_score)

import tensorflow as tf
import keras

random.seed(SEED); np.random.seed(SEED)
keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception as e:
    print("Could not enable op determinism:", e)

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

# ---------------- Settings you may change ----------------
VERSION = "1.0.0"                                   # bump when you retrain
DATA_PATH = os.environ.get("PM_DATA_PATH", "/content/motor_predictive_maintenance_master.csv")
OUTPUT_ROOT = Path(os.environ.get("PM_OUT", "/content/outputs"))
TIMESTAMP_FORMAT = "%m/%d/%Y %H:%M"                # month-first. Evidence: inspection report (day>12 only in 2nd field). Confirm with data owner.
FAST_DEBUG = os.environ.get("PM_FAST_TEST") == "1" # True shrinks tuning to a smoke test. Never use for real results.
# ---------------------------------------------------------
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Python      :", sys.version.split()[0])
print("numpy       :", np.__version__, "| pandas:", pd.__version__, "| scikit-learn:", sklearn.__version__)
print("tensorflow  :", tf.__version__, "| keras:", keras.__version__)
print("GPU devices :", tf.config.list_physical_devices("GPU") or "none (CPU is fine for this dataset)")
if FAST_DEBUG:
    print("\n*** FAST_DEBUG is ON: results are NOT valid for reporting ***")

### Label-provenance gate (fill this in)
**Why this matters:** every metric in this notebook measures agreement with `condition_label`. If you do not know *who or what* assigned those labels, the metrics measure agreement with an unknown process. The notebook will still run (so you can learn the workflow), but every output, the model card and the bundle will be stamped **PROVISIONAL** until `documented` is `True`.

Answer these from the data owner or supervisor, in writing, and keep the evidence:
* Who/what assigned `Normal`, `Alert`, `Danger` (engineer inspection, a standard such as a vibration-severity chart, maintenance records, an automatic alarm)?
* Were the labels computed **from these same sensor columns**? (If yes, set `assigned_from_same_sensor_columns = True`.)
* On what date range and by whom?

In [ ]:
LABEL_PROVENANCE = {
    "documented": False,                         # set True ONLY when you have written evidence
    "assigned_by": "TODO: e.g. maintenance engineer / vibration-severity rule / alarm system",
    "rule_or_standard": "TODO: name the rule or standard, or write 'none, human judgement'",
    "assigned_from_same_sensor_columns": None,   # True / False once known
    "evidence_reference": "TODO: email, document, or meeting date",
}
PROVISIONAL = not LABEL_PROVENANCE["documented"]
STAMP = ("PROVISIONAL: label provenance is NOT documented. Results describe agreement with an unverified label."
         if PROVISIONAL else "Label provenance documented (see model card).")
print(STAMP)

## 3. Load the dataset (strict, fail-loudly loader)

The previous step's cleaning module is not finished, so this loader does the minimum needed and **stops with a clear message** if anything differs from what the inspection report showed. It:
* reads **everything as text first** (so nothing is silently converted);
* removes the sub-heading row **only after** checking that its 15 cells match the expected names exactly;
* converts numbers and timestamps **strictly** and refuses to continue if any non-empty cell fails to convert;
* never fills, drops or invents a measurement, timestamp or label.

The original file is only **read**. Its SHA-256 fingerprint is stored so you can prove it was not altered.

In [ ]:
class DataSchemaError(Exception):
    """Raised when the file does not look like the file we inspected."""

ID_HEADERS = ["Record ID", "Motor ID", "Measurement timestamps"]
# raw sub-heading -> clean name, unit, description (positions: DE/NDE = drive end / non-drive end; H/V = 'Header'/'Vrtex' as typed in the file)
SENSOR_COLUMNS = {
    "DE_H_mm/s":   ("de_h_velocity_mm_s",  "mm/s", "Drive end, 'Header' (assumed horizontal): vibration velocity"),
    "DE_H_gE":     ("de_h_ge",             "gE",   "Drive end, 'Header': gE reading"),
    "DE_H_temp_C": ("de_h_temp_c",         "degC", "Drive end, 'Header': temperature"),
    "DE_V_mm/s":   ("de_v_velocity_mm_s",  "mm/s", "Drive end, 'Vrtex' (assumed vertical): vibration velocity"),
    "DE_V_gE":     ("de_v_ge",             "gE",   "Drive end, 'Vrtex': gE reading"),
    "DE_V_temp_C": ("de_v_temp_c",         "degC", "Drive end, 'Vrtex': temperature"),
    "NDE_H_mm/s":  ("nde_h_velocity_mm_s", "mm/s", "Non-drive end, 'Header': vibration velocity"),
    "NDE_H_gE":    ("nde_h_ge",            "gE",   "Non-drive end, 'Header': gE reading"),
    "NDE_H_temp_C":("nde_h_temp_c",        "degC", "Non-drive end, 'Header': temperature"),
    "NDE_V_mm/s":  ("nde_v_velocity_mm_s", "mm/s", "Non-drive end, 'Vrtex': vibration velocity"),
    "NDE_V_gE":    ("nde_v_ge",            "gE",   "Non-drive end, 'Vrtex': gE reading"),
    "NDE_V_temp_C":("nde_v_temp_c",        "degC", "Non-drive end, 'Vrtex': temperature"),
    "rotational_speed_rpm": ("rotational_speed_rpm", "rpm", "Rotational speed (measured or nominal: unconfirmed)"),
    "load_percent":         ("load_percent",         "%",   "Motor load (measured or nominal: unconfirmed)"),
}
LABEL_RAW = "condition_label"
CLASS_NAMES = ["Normal", "Alert", "Danger"]      # ORDER matters: 0 < 1 < 2 (increasing severity)
CLASS_TO_ID = {c: i for i, c in enumerate(CLASS_NAMES)}
EXPECTED_SUBHEADINGS = list(SENSOR_COLUMNS) + [LABEL_RAW]
NUMERIC_CLEAN = [v[0] for v in SENSOR_COLUMNS.values()]

def load_raw_motor_csv(path):
    """Return (clean_df, report_dict). Raises DataSchemaError on any unexpected structure."""
    raw_bytes = Path(path).read_bytes()
    report = {"file_name": Path(path).name, "sha256": hashlib.sha256(raw_bytes).hexdigest(), "bytes": len(raw_bytes)}
    grid = pd.read_csv(path, header=None, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    report["raw_shape"] = list(grid.shape)
    if grid.shape[1] != 18:
        raise DataSchemaError(f"Expected 18 columns, found {grid.shape[1]}.")
    row0 = [str(x).strip() for x in grid.iloc[0]]
    row1 = [str(x).strip() for x in grid.iloc[1]]
    if row0[:3] != ID_HEADERS:
        raise DataSchemaError(f"Row 0 identifier headers differ. Expected {ID_HEADERS}, found {row0[:3]}.")
    if row1[3:] != EXPECTED_SUBHEADINGS:
        diffs = [(i + 3, a, b) for i, (a, b) in enumerate(zip(EXPECTED_SUBHEADINGS, row1[3:])) if a != b]
        raise DataSchemaError(f"Sub-heading row differs from expectation at (column, expected, found): {diffs}")
    if any(row1[:3]):
        raise DataSchemaError("Identifier cells in the sub-heading row should be empty.")
    # group names must sit over the right columns (guards against a shifted sensor block)
    for col, must_start in {3: "drive end", 6: "drive end", 9: "non drive end", 12: "non drive end"}.items():
        if not row0[col].lower().startswith(must_start):
            raise DataSchemaError(f"Group header at column {col} is '{row0[col]}', expected it to start with '{must_start}'.")
    report["group_headers_row0"] = {c: row0[c] for c in (3, 6, 9, 12)}
    report["subheading_row_removed"] = True

    body = grid.iloc[2:].copy().reset_index(drop=True)
    body.columns = ["record_id", "motor_id", "measured_at_raw"] + EXPECTED_SUBHEADINGS
    body = body.apply(lambda s: s.str.strip())
    report["data_rows"] = int(len(body))

    out = pd.DataFrame({"record_id": body["record_id"], "motor_id": body["motor_id"]})
    ts = pd.to_datetime(body["measured_at_raw"].replace("", pd.NA), format=TIMESTAMP_FORMAT, errors="coerce")
    bad_ts = int((ts.isna() & (body["measured_at_raw"] != "")).sum())
    if bad_ts:
        raise DataSchemaError(f"{bad_ts} timestamps do not match format {TIMESTAMP_FORMAT}: "
                              f"{body.loc[ts.isna() & (body['measured_at_raw'] != ''), 'measured_at_raw'].head(5).tolist()}")
    out["measured_at"] = ts
    for raw_name, (clean, _, _) in SENSOR_COLUMNS.items():
        num = pd.to_numeric(body[raw_name].replace("", pd.NA), errors="coerce")
        bad = body.loc[num.isna() & (body[raw_name] != ""), raw_name]
        if len(bad):
            raise DataSchemaError(f"Column {raw_name}: non-numeric text found: {bad.value_counts().head(5).to_dict()}")
        out[clean] = num.astype("float64")
    labels = body[LABEL_RAW].replace("", pd.NA)
    unknown = sorted(set(labels.dropna()) - set(CLASS_NAMES))
    if unknown:
        raise DataSchemaError(f"Unexpected label values: {unknown}. Allowed: {CLASS_NAMES}")
    out["condition_label"] = labels
    return out, report

# ---- locate the file (upload helper for Colab) ----
if not Path(DATA_PATH).exists():
    try:
        from google.colab import files
        print("File not found at", DATA_PATH, "\nPlease choose motor_predictive_maintenance_master.csv ...")
        up = files.upload()
        DATA_PATH = "/content/" + next(iter(up))
    except ImportError:
        raise FileNotFoundError(f"Set DATA_PATH to your CSV. Not found: {DATA_PATH}")

df, load_report = load_raw_motor_csv(DATA_PATH)
# stable chronological order within each motor (ties broken by record_id; nothing is dropped)
df = df.sort_values(["motor_id", "measured_at", "record_id"], kind="stable").reset_index(drop=True)
print("Original file SHA-256:", load_report["sha256"])
print("Rows loaded:", len(df), "| columns:", df.shape[1], "| sub-heading row removed after exact match check")
df.head()

## 4. Validate the schema and data types
These are hard checks. If one fails the notebook stops, because a wrong assumption here would silently corrupt everything after it.

In [ ]:
assert df["record_id"].is_unique, "record_id must be unique"
assert df["motor_id"].ne("").all() and df["motor_id"].notna().all(), "motor_id must not be empty"
assert set(df["condition_label"].dropna()) <= set(CLASS_NAMES)
assert str(df["measured_at"].dtype).startswith("datetime64")
assert all(df[c].dtype == "float64" for c in NUMERIC_CLEAN)

schema_table = pd.DataFrame({
    "column": df.columns, "dtype": [str(t) for t in df.dtypes],
    "non_null": df.notna().sum().values, "n_unique": df.nunique().values})
display(schema_table)
print(f"\nSchema OK. {df['motor_id'].nunique()} motors | {len(df)} records | "
      f"{df['measured_at'].min():%Y-%m-%d} to {df['measured_at'].max():%Y-%m-%d}")

## 5. Exploratory data analysis: overview

In [ ]:
display(df[NUMERIC_CLEAN].describe().T.round(3))
print("\nThese are descriptive statistics only. Observed minimum/maximum values are NOT validity limits or alarm thresholds.")

## 6. Missing-value and duplicate analysis
Policy: **we never impute silently.** If any model input is missing, the notebook stops so you can decide, and document, what to do.

In [ ]:
miss = df.isna().sum().to_frame("missing"); miss["percent"] = (miss["missing"] / len(df) * 100).round(3)
display(miss[miss["missing"] > 0] if miss["missing"].any() else pd.DataFrame({"result": ["no missing values in any column"]}))

dup_full = int(df.drop(columns=["record_id"]).duplicated().sum())
dup_motor_time = int(df.duplicated(["motor_id", "measured_at"], keep=False).sum())
dup_features = int(df.duplicated(NUMERIC_CLEAN, keep=False).sum())
print(f"Duplicate rows ignoring record_id          : {dup_full}")
print(f"Rows sharing the same motor + timestamp    : {dup_motor_time}")
print(f"Rows with identical sensor/operating values: {dup_features}")
if dup_motor_time:
    display(df[df.duplicated(["motor_id", "measured_at"], keep=False)].head(10))
    print("Same motor+timestamp may be a logging issue. Reported only: nothing was dropped.")

features_missing = int(df[NUMERIC_CLEAN].isna().sum().sum())
assert features_missing == 0 and df["condition_label"].notna().all(), (
    "Missing model inputs or labels found. Decide on a documented policy (e.g. exclude those rows and report the count). Do NOT impute silently.")

## 7. Condition-class distribution

In [ ]:
cls_counts = df["condition_label"].value_counts().reindex(CLASS_NAMES)
display(pd.DataFrame({"records": cls_counts, "percent": (cls_counts / len(df) * 100).round(1)}))
imbalance_ratio = float(cls_counts.max() / cls_counts.min())
print(f"Imbalance ratio (largest class / smallest class): {imbalance_ratio:.1f} : 1")
print(f"A model that always answers '{cls_counts.idxmax()}' would score {cls_counts.max()/len(df):.1%} accuracy while detecting 0% of the other classes.")

fig, ax = plt.subplots(figsize=(5, 3.2))
ax.bar(CLASS_NAMES, cls_counts.values, color=["#1f7547", "#955700", "#b42318"])
for i, v in enumerate(cls_counts.values): ax.text(i, v, str(v), ha="center", va="bottom")
ax.set_ylabel("records"); ax.set_title("condition_label distribution"); plt.show()

comp = pd.crosstab(df["motor_id"], df["condition_label"]).reindex(columns=CLASS_NAMES, fill_value=0)
print(f"\nMotors with at least one Alert  record: {(comp['Alert'] > 0).sum()} of {len(comp)}")
print(f"Motors with at least one Danger record: {(comp['Danger'] > 0).sum()} of {len(comp)}")
print("If few motors supply most Danger records, a motor-grouped split can leave a split with too few of them (handled in Section 11).")

## 8. Sensor distributions and trends over time

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(14, 10)); axes = axes.ravel()
for ax, c in zip(axes, NUMERIC_CLEAN):
    ax.hist(df[c], bins=40, color="#0e6674", alpha=0.85); ax.set_title(c, fontsize=9)
for ax in axes[len(NUMERIC_CLEAN):]: ax.axis("off")
fig.suptitle("Distribution of every model input", y=1.0); fig.tight_layout(); plt.show()

In [ ]:
groups = {"Velocity (mm/s)": [c for c in NUMERIC_CLEAN if "velocity" in c],
          "gE": [c for c in NUMERIC_CLEAN if c.endswith("_ge")],
          "Temperature (degC)": [c for c in NUMERIC_CLEAN if "temp" in c]}
fig, axes = plt.subplots(3, 4, figsize=(14, 9))
for r, (gname, cols) in enumerate(groups.items()):
    for k, c in enumerate(cols):
        data = [df.loc[df["condition_label"] == cl, c] for cl in CLASS_NAMES]
        try: axes[r, k].boxplot(data, tick_labels=CLASS_NAMES, showfliers=False)
        except TypeError: axes[r, k].boxplot(data, labels=CLASS_NAMES, showfliers=False)   # older matplotlib
        axes[r, k].set_title(c, fontsize=9)
fig.suptitle("Each sensor by condition class (outliers hidden for readability)"); fig.tight_layout(); plt.show()

In [ ]:
# Trends over time: the 3 motors with the most Danger records (ties by name, so the choice is reproducible).
order = comp.assign(m=comp.index).sort_values(["Danger", "Alert", "m"], ascending=[False, False, True])
show = list(order.index[:3])
colors = {"Normal": "#1f7547", "Alert": "#955700", "Danger": "#b42318"}
fig, axes = plt.subplots(len(show), 1, figsize=(11, 3.1 * len(show)), sharex=False)
axes = np.atleast_1d(axes)
for ax, m in zip(axes, show):
    d = df[df["motor_id"] == m]
    ax.plot(d["measured_at"], d["de_h_velocity_mm_s"], color="#999", lw=0.8)
    ax.scatter(d["measured_at"], d["de_h_velocity_mm_s"], c=d["condition_label"].map(colors), s=14)
    ax.set_title(f"{m}: de_h_velocity_mm_s over time (colour = condition_label)", fontsize=9); ax.set_ylabel("mm/s")
plt.tight_layout(); plt.show()

In [ ]:
corr = df[NUMERIC_CLEAN].corr()
fig, ax = plt.subplots(figsize=(8, 6.5)); im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90, fontsize=7)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns, fontsize=7); ax.grid(False)
plt.colorbar(im); ax.set_title("Correlation between inputs"); plt.show()
print("Strongly correlated inputs carry overlapping information. This is normal for sensors at neighbouring positions.")

## 9. Motor-level and timestamp-level analysis
Questions answered here: how many readings per motor, over what period, how regular is the sampling, and does a motor's label ever *improve* (which could mean a repair happened)?

In [ ]:
sev = df["condition_label"].map(CLASS_TO_ID)
def motor_summary(d):
    s = d["condition_label"].map(CLASS_TO_ID).to_numpy()
    steps = np.diff(s)
    gaps = d["measured_at"].diff().dt.total_seconds().dropna() / 86400
    return pd.Series({"records": len(d), "first": d["measured_at"].min(), "last": d["measured_at"].max(),
                      "span_days": (d["measured_at"].max() - d["measured_at"].min()).days,
                      "median_gap_days": gaps.median() if len(gaps) else np.nan,
                      "n_alert": int((s == 1).sum()), "n_danger": int((s == 2).sum()),
                      "label_worsenings": int((steps > 0).sum()), "label_improvements": int((steps < 0).sum()),
                      "rpm_median": d["rotational_speed_rpm"].median()})
motor_tbl = pd.DataFrame({m: motor_summary(d) for m, d in df.groupby("motor_id")}).T
for c_ in ["records","span_days","median_gap_days","n_alert","n_danger","label_worsenings","label_improvements","rpm_median"]: motor_tbl[c_] = pd.to_numeric(motor_tbl[c_])
display(motor_tbl.head(12))
print(f"\nRecords per motor: min {int(motor_tbl.records.min())}, median {motor_tbl.records.median():.0f}, max {int(motor_tbl.records.max())}")
print(f"Observation span: {df['measured_at'].min():%Y-%m-%d} to {df['measured_at'].max():%Y-%m-%d}")
print(f"Median gap between consecutive readings of a motor: {motor_tbl.median_gap_days.median():.1f} days")
print(f"Motors whose label ever improves (e.g. Danger -> Normal): {(motor_tbl.label_improvements > 0).sum()} of {len(motor_tbl)}")
print("Improvements can mean a repair or a re-labelling. Ask the data owner: they matter for any future-failure definition.")

fig, ax = plt.subplots(figsize=(10, 3.2))
for i, (m, d) in enumerate(df.groupby("motor_id")): ax.plot([d["measured_at"].min(), d["measured_at"].max()], [i, i], color="#0e6674", lw=2)
ax.set_xlabel("time"); ax.set_ylabel("motor index"); ax.set_title("Time coverage of each motor"); plt.show()

gaps_all = df.groupby("motor_id")["measured_at"].diff().dt.total_seconds().dropna() / 86400
plt.figure(figsize=(6, 3)); plt.hist(gaps_all.clip(upper=gaps_all.quantile(0.99)), bins=40, color="#0e6674"); plt.xlabel("days between consecutive readings"); plt.title("Sampling intervals"); plt.show()

In [ ]:
# Can operating conditions identify a motor? If rpm barely varies within a motor but differs between motors,
# rpm acts like a motor ID, which weakens a motor-grouped evaluation and should be discussed.
within = df.groupby("motor_id")["rotational_speed_rpm"].std().median()
between = df.groupby("motor_id")["rotational_speed_rpm"].median().std()
print(f"rpm: typical spread WITHIN a motor = {within:.1f} | spread of motor medians BETWEEN motors = {between:.1f}")
within_l = df.groupby("motor_id")["load_percent"].std().median(); between_l = df.groupby("motor_id")["load_percent"].median().std()
print(f"load_percent: within = {within_l:.1f} | between = {between_l:.1f}")
print("A much larger between-motor than within-motor spread means that column partly encodes WHICH motor it is (motor type).")

## 10. Define input features and the target

**Inputs (14):** 12 sensor readings (velocity, gE, temperature at four positions) + `rotational_speed_rpm` + `load_percent`.

**Deliberately excluded** (and checked by code so nobody adds them by accident):
| Column | Reason |
|---|---|
| `condition_label` | It is the **target** |
| `record_id` | Row identifier; carries no physical meaning and can leak file order |
| `motor_id` | Lets a model memorise individual motors instead of learning condition |
| `measured_at` | Calendar time is not a physical cause of condition and would encourage memorising periods |

Operating conditions (rpm, load) are legitimate inputs because the same vibration level means different things at different loads. Their exact meaning (measured vs nominal) is still to be confirmed.

In [ ]:
FEATURES = list(NUMERIC_CLEAN)                      # this ORDER is the contract with the backend
TARGET = "condition_label"
EXCLUDED = {
    "condition_label": "target",
    "record_id": "row identifier",
    "motor_id": "identifies the motor (memorisation risk)",
    "measured_at": "calendar time (memorisation risk)",
}
FEATURE_META = {clean: {"unit": unit, "description": desc} for _, (clean, unit, desc) in SENSOR_COLUMNS.items()}

assert len(FEATURES) == 14 and len(set(FEATURES)) == 14
assert not (set(FEATURES) & set(EXCLUDED)), "an excluded column leaked into FEATURES"
assert not any(w in f for f in FEATURES for w in ("label", "record", "motor_id", "measured_at", "split"))
df["y"] = df[TARGET].map(CLASS_TO_ID).astype(int)
print("Feature order:"); [print(f"  {i:2d}. {f:24s} [{FEATURE_META[f]['unit']}]") for i, f in enumerate(FEATURES)]
print("\nClass mapping:", CLASS_TO_ID)

## 11. Leakage-resistant training / validation / test split

### The problem
One motor produces ~48 readings that look very alike (same machine, same mounting, same environment). If some of a motor's readings are in training and others in testing, the model can "recognise the motor" and look accurate without learning anything general. That is **leakage through repeated measurements**.

### The choice: motor-grouped split
All readings of a motor go to **exactly one** of train / validation / test. This matches the question *"does it work on a motor it has never seen?"*

| | Motor-grouped split (chosen) | Chronological split (alternative) |
|---|---|---|
| Question answered | Generalisation to **unseen motors** | Generalisation to the **future of known motors** |
| Strength | Directly tests deployment on new machines | Mimics real forecasting; needs real failure events to be meaningful |
| Weakness | Only ~40 motors, so the test set holds few motors and scores are uncertain | Same motors in train and test, so it cannot tell you about new motors |
| When to prefer | Condition classification (this notebook, Option A) | Future-failure prediction (Option B) with verified events |

### Making the split fair without peeking
With few motors and a rare `Danger` class, a random motor partition can leave one split with almost no `Danger` rows. The code therefore tries many seeded random partitions of **motors** and keeps the one whose class mix and size best match the whole dataset, requiring a minimum count of every class in every split. The search looks **only at label counts**, never at features or model results, so it cannot favour a split that makes the model look good.

In [ ]:
SPLIT_FRACTIONS = (0.60, 0.20, 0.20)     # train / validation / test, by number of MOTORS
MIN_CLASS_RECORDS = 10                   # every class needs at least this many records in every split
N_SPLIT_TRIALS = 500 if FAST_DEBUG else 5000

def propose_grouped_split(frame, seed):
    motors = np.array(sorted(frame["motor_id"].unique()))
    M = (pd.crosstab(frame["motor_id"], frame["y"]).reindex(index=motors, columns=[0, 1, 2], fill_value=0)).to_numpy()
    total = M.sum(0); glob = total / total.sum()
    n = len(motors); n_tr = int(round(SPLIT_FRACTIONS[0] * n)); n_va = int(round(SPLIT_FRACTIONS[1] * n))
    rng = np.random.default_rng(seed); best = None; feasible = 0
    for _ in range(N_SPLIT_TRIALS):
        perm = rng.permutation(n)
        parts = [perm[:n_tr], perm[n_tr:n_tr + n_va], perm[n_tr + n_va:]]
        score, ok = 0.0, True
        for k, idx in enumerate(parts):
            c = M[idx].sum(0)
            if (c < MIN_CLASS_RECORDS).any(): ok = False; break
            score += np.abs(c / c.sum() - glob).sum() + abs(c.sum() / total.sum() - SPLIT_FRACTIONS[k])
        if ok:
            feasible += 1
            if best is None or score < best[0]: best = (score, parts)
    if best is None:
        raise RuntimeError(f"No motor partition gives >= {MIN_CLASS_RECORDS} records of every class in every split. "
                           "The rare class is concentrated in too few motors. Lower MIN_CLASS_RECORDS (and say so in your report) "
                           "or collect more data. Do NOT silently switch to a row-level split.")
    return {name: list(motors[idx]) for name, idx in zip(["train", "val", "test"], best[1])}, feasible

split_motors, n_feasible = propose_grouped_split(df, SEED)
df["split"] = df["motor_id"].map({m: s for s, ms in split_motors.items() for m in ms})

# ---- hard guarantees ----
sets = {s: set(v) for s, v in split_motors.items()}
assert sets["train"].isdisjoint(sets["val"]) and sets["train"].isdisjoint(sets["test"]) and sets["val"].isdisjoint(sets["test"])
assert df["split"].notna().all() and sum(len(v) for v in sets.values()) == df["motor_id"].nunique()
train, val, test = (df[df.split == s].copy() for s in ["train", "val", "test"])

comp_tbl = pd.DataFrame({s: {"motors": len(split_motors[s]), "records": int((df.split == s).sum()),
                             **{c: int(((df.split == s) & (df.condition_label == c)).sum()) for c in CLASS_NAMES}} for s in split_motors}).T
comp_tbl["Danger %"] = (comp_tbl["Danger"] / comp_tbl["records"] * 100).round(1)
display(comp_tbl)
print(f"{n_feasible} of {N_SPLIT_TRIALS} candidate partitions satisfied the minimum-class rule; the best one by class-mix similarity was kept.")
print("No motor appears in more than one split: verified by assertion.")

### 11b. Demonstration: how much does naive splitting inflate the score?
Uses **only training + validation motors** (the test motors are not touched). The same Random Forest is scored by cross-validation twice: once splitting **rows** at random (leaky), once splitting by **motor** (honest). A large gap is the evidence for your choice of a grouped split, and it is a strong viva exhibit.

In [ ]:
from sklearn.model_selection import cross_val_predict
pool = df[df.split.isin(["train", "val"])]
rf_demo = RandomForestClassifier(n_estimators=60 if FAST_DEBUG else 200, random_state=SEED, n_jobs=-1)
Xp, yp, gp = pool[FEATURES].to_numpy(), pool["y"].to_numpy(), pool["motor_id"].to_numpy()
pred_naive = cross_val_predict(rf_demo, Xp, yp, cv=StratifiedKFold(5, shuffle=True, random_state=SEED))
pred_group = cross_val_predict(rf_demo, Xp, yp, cv=StratifiedGroupKFold(5, shuffle=True, random_state=SEED), groups=gp)
leak_demo = pd.DataFrame({"macro_F1": [f1_score(yp, pred_naive, average="macro"), f1_score(yp, pred_group, average="macro")],
                          "accuracy": [accuracy_score(yp, pred_naive), accuracy_score(yp, pred_group)]},
                         index=["random row split (leaky)", "motor-grouped split (honest)"]).round(3)
display(leak_demo)
LEAKAGE_DEMO = leak_demo.to_dict()

## 12. Fit preprocessing on training data only
**Standardisation** rescales each input to mean 0 and standard deviation 1 (`(x − mean) / std`). Neural networks train poorly when inputs have very different scales (rpm ≈ 1500, gE ≈ 0.3). The mean and std are learned from the **training motors only**. Validation and test data are *transformed with those numbers*; they never influence them. Computing them on all data would let test information leak into training.

In [ ]:
X_train = train[FEATURES].to_numpy(dtype="float64"); y_train = train["y"].to_numpy()
X_val   = val[FEATURES].to_numpy(dtype="float64");   y_val   = val["y"].to_numpy()
X_test  = test[FEATURES].to_numpy(dtype="float64");  y_test  = test["y"].to_numpy()
g_val, g_test = val["motor_id"].to_numpy(), test["motor_id"].to_numpy()

scaler = StandardScaler().fit(X_train)            # <- the ONLY place preprocessing is fitted
assert scaler.n_samples_seen_ == len(X_train), "scaler saw more than the training data"
X_train_s, X_val_s, X_test_s = (scaler.transform(a) for a in (X_train, X_val, X_test))
assert np.allclose(X_train_s.mean(0), 0, atol=1e-9)

shift = pd.DataFrame({"val_mean_in_train_SD": X_val_s.mean(0), "test_mean_in_train_SD": X_test_s.mean(0)}, index=FEATURES).round(2)
display(shift)
print("Values far from 0 show that unseen motors differ from training motors: this is the generalisation challenge.")
if np.isnan(X_train_s).any() or np.isnan(X_val_s).any() or np.isnan(X_test_s).any(): raise ValueError("NaN after scaling")

### 12b. Label-rule probe (uses training and validation data only)
**Question:** can very simple rules on one sensor type reproduce the labels almost perfectly? If a depth-2 decision tree on **velocity alone** already reaches near-perfect validation macro-F1, the labels may have been *derived from velocity thresholds*. Then a high ANN score mostly shows that the network rediscovered a known rule, not that it detects motor deterioration.

This is a **diagnostic, not proof**: confirm with the label provenance from the data owner (Section 2).

In [ ]:
VEL = [c for c in FEATURES if "velocity" in c]; GE = [c for c in FEATURES if c.endswith("_ge")]
TMP = [c for c in FEATURES if "temp" in c]; OPS = ["rotational_speed_rpm", "load_percent"]
probe_rows = []
for name, cols in {"velocity only": VEL, "gE only": GE, "temperature only": TMP, "operating only": OPS, "all 14 inputs": FEATURES}.items():
    for depth in (1, 2, 3):
        t = DecisionTreeClassifier(max_depth=depth, class_weight="balanced", random_state=SEED).fit(train[cols], y_train)
        p = t.predict(val[cols])
        probe_rows.append({"inputs": name, "tree_depth": depth, "val_macro_F1": f1_score(y_val, p, average="macro", labels=[0, 1, 2], zero_division=0),
                           "val_accuracy": accuracy_score(y_val, p)})
probe = pd.DataFrame(probe_rows).round(3)
display(probe.pivot(index="inputs", columns="tree_depth", values="val_macro_F1"))
t2 = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=SEED).fit(train[VEL], y_train)
print("Depth-2 tree learned from VELOCITY inputs only (training data):\n")
print(export_text(t2, feature_names=VEL))
LABEL_PROBE = probe.to_dict("records")
best_vel = probe[(probe.inputs == "velocity only")].val_macro_F1.max()
print(f"Best validation macro-F1 from a tiny velocity-only tree: {best_vel:.3f}")
if best_vel >= 0.9:
    print("WARNING: labels are almost reproducible from velocity alone. Strongly check whether they were assigned by a velocity rule.")

## 13. Train the Artificial Neural Network (primary model)

**Architecture:** a fully connected network (multilayer perceptron): 14 inputs → one to three hidden layers (ReLU) → 3 outputs (softmax = probability of Normal / Alert / Danger). With only ~1,100 training rows a small network is appropriate; large networks would simply memorise.

**Training:** the Adam optimiser minimises cross-entropy. **Early stopping** watches the validation loss and restores the best epoch, which prevents over-training. Because validation data steers early stopping *and* model selection, validation scores are slightly optimistic by design; only the untouched test set gives an unbiased estimate.

**Declared in advance:** the ANN is the research model. Baselines are comparators and will not be swapped in after seeing test results.

In [ ]:
LABELS = [0, 1, 2]
MAX_EPOCHS = 8 if FAST_DEBUG else 300
PATIENCE = 3 if FAST_DEBUG else 20

# ---------- shared evaluation helpers ----------
def compute_metrics(y_true, proba):
    y_true = np.asarray(y_true); proba = np.asarray(proba); y_pred = proba.argmax(1)
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=LABELS, zero_division=0)
    m = {"n": int(len(y_true)),
         "accuracy": float(accuracy_score(y_true, y_pred)),
         "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
         "macro_f1": float(f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0)),
         "weighted_f1": float(f1_score(y_true, y_pred, labels=LABELS, average="weighted", zero_division=0)),
         "macro_precision": float(p.mean()), "macro_recall": float(r.mean()),
         "mcc": float(matthews_corrcoef(y_true, y_pred)),
         "weighted_kappa_quadratic": float(cohen_kappa_score(y_true, y_pred, weights="quadratic", labels=LABELS)),
         "log_loss": float(log_loss(y_true, np.clip(proba, 1e-12, 1), labels=LABELS)),
         "per_class": {CLASS_NAMES[i]: {"precision": float(p[i]), "recall": float(r[i]), "f1": float(f[i]), "support": int(s[i])} for i in LABELS},
         "confusion_matrix": confusion_matrix(y_true, y_pred, labels=LABELS).tolist()}
    if len(np.unique(y_true)) == len(LABELS):
        m["roc_auc_ovr_macro"] = float(roc_auc_score(y_true, proba, multi_class="ovr", average="macro", labels=LABELS))
        m["pr_auc_macro"] = float(np.mean([average_precision_score((y_true == k).astype(int), proba[:, k]) for k in LABELS]))
    else:
        m["roc_auc_ovr_macro"] = m["pr_auc_macro"] = None
    nn_t, nn_p = y_true > 0, y_pred > 0     # "any abnormal condition" view derived from the 3-class prediction
    m["abnormal_recall"] = float(recall_score(nn_t, nn_p, zero_division=0))
    m["abnormal_precision"] = float(precision_score(nn_t, nn_p, zero_division=0))
    m["danger_predicted_as_normal"] = int(((y_true == 2) & (y_pred == 0)).sum())
    return m

def plot_cm(ax, cm, title, normalise):
    cm = np.array(cm, dtype=float)
    if normalise: cm = cm / np.clip(cm.sum(1, keepdims=True), 1, None)
    ax.imshow(cm, cmap="Blues", vmin=0, vmax=1 if normalise else max(cm.max(), 1)); ax.grid(False)
    ax.set_xticks(range(3)); ax.set_xticklabels(CLASS_NAMES); ax.set_yticks(range(3)); ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("predicted"); ax.set_ylabel("actual"); ax.set_title(title, fontsize=10)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{cm[i, j]:.2f}" if normalise else f"{int(cm[i, j])}", ha="center", va="center",
                    color="white" if cm[i, j] > (0.5 if normalise else cm.max() / 2) else "black")

# ---------- ANN ----------
cw = compute_class_weight("balanced", classes=np.array(LABELS), y=y_train)
CLASS_WEIGHT_BALANCED = {i: float(w) for i, w in enumerate(cw)}
print("'balanced' class weights computed from TRAINING labels only:", {CLASS_NAMES[i]: round(w, 2) for i, w in CLASS_WEIGHT_BALANCED.items()})

def build_ann(n_features, cfg):
    reg = keras.regularizers.l2(cfg["l2"]) if cfg["l2"] else None
    layers = [keras.Input(shape=(n_features,))]
    for _ in range(cfg["hidden_layers"]):
        layers.append(keras.layers.Dense(cfg["units"], activation="relu", kernel_regularizer=reg))
        if cfg["dropout"]: layers.append(keras.layers.Dropout(cfg["dropout"]))
    layers.append(keras.layers.Dense(len(CLASS_NAMES), activation="softmax"))
    model = keras.Sequential(layers)
    model.compile(optimizer=keras.optimizers.Adam(cfg["lr"]), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

def train_ann(cfg, seed):
    keras.utils.set_random_seed(seed)
    model = build_ann(X_train_s.shape[1], cfg)
    stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True)
    hist = model.fit(X_train_s, y_train, validation_data=(X_val_s, y_val), epochs=MAX_EPOCHS, batch_size=cfg["batch_size"],
                     class_weight=CLASS_WEIGHT_BALANCED if cfg["class_weight"] == "balanced" else None,
                     callbacks=[stop], verbose=0, shuffle=True)
    best_epoch = int(np.argmin(hist.history["val_loss"])) + 1
    return model, hist.history, best_epoch

DEFAULT_CFG = dict(hidden_layers=2, units=32, dropout=0.1, l2=1e-4, lr=1e-3, batch_size=32, class_weight="balanced")
ann_default, hist_default, ep_default = train_ann(DEFAULT_CFG, SEED)
ann_default.summary()

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(hist_default["loss"], label="train"); ax[0].plot(hist_default["val_loss"], label="validation")
ax[0].axvline(ep_default - 1, color="grey", ls="--", label="best epoch"); ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(hist_default["accuracy"], label="train"); ax[1].plot(hist_default["val_accuracy"], label="validation"); ax[1].set_title("Accuracy"); ax[1].legend()
plt.tight_layout(); plt.show()
print("A growing gap between the curves means over-fitting; early stopping returns the weights from the best epoch.")

## 14. Baseline models for comparison
A result is only meaningful relative to something. Three baselines, all fitted on the same standardised training data:
* **Dummy (most frequent class):** the "do nothing clever" floor. It shows how misleading plain accuracy is when classes are imbalanced.
* **Logistic Regression:** a simple linear model. If the ANN cannot beat it, the extra complexity is not justified.
* **Random Forest:** a strong, standard tabular model; the one the ANN most needs to be compared with.

In [ ]:
baseline_default = {
    "Dummy (most frequent)": DummyClassifier(strategy="most_frequent").fit(X_train_s, y_train),
    "LogisticRegression": LogisticRegression(max_iter=3000, random_state=SEED).fit(X_train_s, y_train),
    "RandomForest": RandomForestClassifier(n_estimators=50 if FAST_DEBUG else 300, random_state=SEED, n_jobs=-1).fit(X_train_s, y_train),
}
for name, mdl in baseline_default.items(): assert list(mdl.classes_) == LABELS

def val_row(name, proba):
    m = compute_metrics(y_val, proba)
    return {"model": name, "accuracy": m["accuracy"], "macro_F1": m["macro_f1"], "balanced_acc": m["balanced_accuracy"],
            "Alert_recall": m["per_class"]["Alert"]["recall"], "Danger_recall": m["per_class"]["Danger"]["recall"]}
rows = [val_row("ANN (default config)", ann_default.predict(X_val_s, verbose=0))] + [val_row(n, m.predict_proba(X_val_s)) for n, m in baseline_default.items()]
display(pd.DataFrame(rows).round(3).set_index("model"))
print("These are VALIDATION scores before tuning. The test set has not been touched.")

## 15. Tune model settings using validation data only
Each model family gets a search over a small, declared grid. Candidates are ranked by **validation macro-F1** (it treats the three classes equally, so a model cannot score well by ignoring `Danger`). The test set is **not** used anywhere in this section.

In [ ]:
ANN_SPACE = {"hidden_layers": [1, 2, 3], "units": [16, 32, 64, 128], "dropout": [0.0, 0.1, 0.3], "l2": [0.0, 1e-4, 1e-3],
             "lr": [1e-3, 3e-3], "batch_size": [32, 64], "class_weight": [None, "balanced"]}
N_ANN_TRIALS = 3 if FAST_DEBUG else 24
ann_trials = []
for i, cfg in enumerate(ParameterSampler(ANN_SPACE, n_iter=N_ANN_TRIALS, random_state=SEED)):
    cfg = dict(cfg)
    mdl, hist, best_ep = train_ann(cfg, SEED)
    m = compute_metrics(y_val, mdl.predict(X_val_s, verbose=0))
    ann_trials.append({"trial": i, **cfg, "best_epoch": best_ep, "val_macro_F1": m["macro_f1"],
                       "val_Danger_recall": m["per_class"]["Danger"]["recall"], "val_accuracy": m["accuracy"], "_model": mdl, "_cfg": cfg})
    print(f"trial {i + 1:2d}/{N_ANN_TRIALS}  macro-F1={m['macro_f1']:.3f}  {cfg}")
ann_trials.sort(key=lambda r: (-r["val_macro_F1"], -r["val_Danger_recall"]))
best_ann = ann_trials[0]; ann_final = best_ann["_model"]; ANN_CFG = best_ann["_cfg"]
display(pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("_")} for r in ann_trials[:5]]).round(3))
print("\nSelected ANN configuration:", ANN_CFG, "| best epoch:", best_ann["best_epoch"])

In [ ]:
# ---- Logistic Regression and Random Forest: validation-only grid search ----
def search_sklearn(make, grid, n_iter=None):
    out = []
    params = list(ParameterGrid(grid))
    if n_iter and len(params) > n_iter:
        params = list(ParameterSampler(grid, n_iter=n_iter, random_state=SEED))
    for p in params:
        mdl = make(**p).fit(X_train_s, y_train)
        m = compute_metrics(y_val, mdl.predict_proba(X_val_s))
        out.append({**p, "val_macro_F1": m["macro_f1"], "val_Danger_recall": m["per_class"]["Danger"]["recall"], "_model": mdl, "_params": p})
    out.sort(key=lambda r: (-r["val_macro_F1"], -r["val_Danger_recall"]))
    return out

lr_trials = search_sklearn(lambda **p: LogisticRegression(max_iter=3000, random_state=SEED, **p),
                           {"C": [0.01, 0.1, 1, 10, 100], "class_weight": [None, "balanced"]})
rf_trials = search_sklearn(lambda **p: RandomForestClassifier(random_state=SEED, n_jobs=-1, **p),
                           {"n_estimators": [50] if FAST_DEBUG else [200, 500], "max_depth": [None, 5, 10],
                            "min_samples_leaf": [1, 3, 5], "class_weight": [None, "balanced_subsample"]}, n_iter=3 if FAST_DEBUG else None)
lr_final, LR_PARAMS = lr_trials[0]["_model"], lr_trials[0]["_params"]
rf_final, RF_PARAMS = rf_trials[0]["_model"], rf_trials[0]["_params"]
dummy_final = baseline_default["Dummy (most frequent)"]
print("Best Logistic Regression:", LR_PARAMS); print("Best Random Forest     :", RF_PARAMS)
display(pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("_")} for r in rf_trials[:3]]).round(3))

In [ ]:
# ---- Seed stability of the selected ANN (validation only) ----
seeds = [SEED + k for k in range(2 if FAST_DEBUG else 5)]
stab = []
for s in seeds:
    mdl, _, _ = train_ann(ANN_CFG, s)
    stab.append(compute_metrics(y_val, mdl.predict(X_val_s, verbose=0))["macro_f1"])
SEED_STABILITY = {"seeds": seeds, "val_macro_f1": stab, "mean": float(np.mean(stab)), "std": float(np.std(stab, ddof=1))}
print("Validation macro-F1 across seeds:", np.round(stab, 3))
print(f"mean = {SEED_STABILITY['mean']:.3f}, sd = {SEED_STABILITY['std']:.3f}")
print("A large spread means the result depends on luck in initialisation: report it, and do not cherry-pick the best seed.")

VAL_SUMMARY = pd.DataFrame([val_row("ANN (selected)", ann_final.predict(X_val_s, verbose=0)),
                            val_row("LogisticRegression (tuned)", lr_final.predict_proba(X_val_s)),
                            val_row("RandomForest (tuned)", rf_final.predict_proba(X_val_s)),
                            val_row("Dummy (most frequent)", dummy_final.predict_proba(X_val_s))]).set_index("model").round(3)
display(VAL_SUMMARY)
print("Tuning is finished. The next section is the ONLY place the test set is used.")

## 16. Evaluate once on the untouched test set
All four finished models are scored **one time** on the test motors. A guard prevents running this twice in a session. If you re-run it after changing anything, the test set stops being "unseen" and your result becomes a validation result. To reproduce, restart the runtime and run all cells again (everything is seeded).

In [ ]:
_TEST_EVALUATED = False
def evaluate_on_test_once():
    global _TEST_EVALUATED
    if _TEST_EVALUATED:
        raise RuntimeError("The test set was already used in this session. Do not tune after looking at it. "
                           "Restart the runtime and run all cells to reproduce.")
    _TEST_EVALUATED = True
    probas = {"ANN": ann_final.predict(X_test_s, verbose=0), "RandomForest": rf_final.predict_proba(X_test_s),
              "LogisticRegression": lr_final.predict_proba(X_test_s), "Dummy (most frequent)": dummy_final.predict_proba(X_test_s)}
    return probas, {k: compute_metrics(y_test, v) for k, v in probas.items()}

test_proba, TEST_METRICS = evaluate_on_test_once()
TEST_EVALUATED_AT = dt.datetime.now(dt.timezone.utc).isoformat()

test_table = pd.DataFrame({k: {"accuracy": m["accuracy"], "balanced_acc": m["balanced_accuracy"], "macro_F1": m["macro_f1"],
                               "weighted_F1": m["weighted_f1"], "Normal_recall": m["per_class"]["Normal"]["recall"],
                               "Alert_recall": m["per_class"]["Alert"]["recall"], "Danger_recall": m["per_class"]["Danger"]["recall"]}
                           for k, m in TEST_METRICS.items()}).T.round(3)
print(STAMP); display(test_table)
print(f"Test set: {len(y_test)} records from {len(split_motors['test'])} motors never used for training or tuning.")

In [ ]:
# Uncertainty: with few test motors the score is uncertain. Bootstrap by MOTOR (resample whole motors with replacement).
B = 100 if FAST_DEBUG else 2000
def cluster_bootstrap(y_true, y_pred, groups, fn, B, seed):
    rng = np.random.default_rng(seed); motors = np.unique(groups)
    idx_by = {m: np.where(groups == m)[0] for m in motors}; vals = []
    for _ in range(B):
        idx = np.concatenate([idx_by[m] for m in rng.choice(motors, len(motors), replace=True)])
        vals.append(fn(y_true[idx], y_pred[idx]))
    return [float(x) for x in np.percentile(vals, [2.5, 97.5])]
mf1 = lambda t, p: f1_score(t, p, labels=LABELS, average="macro", zero_division=0)
drec = lambda t, p: recall_score(t == 2, p == 2, zero_division=0)
CI = {}
for name, pr in test_proba.items():
    yp_ = pr.argmax(1)
    CI[name] = {"macro_f1_95ci": cluster_bootstrap(y_test, yp_, g_test, mf1, B, SEED),
                "danger_recall_95ci": cluster_bootstrap(y_test, yp_, g_test, drec, B, SEED)}
ci_tbl = pd.DataFrame({n: {"macro_F1": TEST_METRICS[n]["macro_f1"], "macro_F1 95% CI": f"[{c['macro_f1_95ci'][0]:.3f}, {c['macro_f1_95ci'][1]:.3f}]",
                           "Danger_recall": TEST_METRICS[n]["per_class"]["Danger"]["recall"],
                           "Danger_recall 95% CI": f"[{c['danger_recall_95ci'][0]:.3f}, {c['danger_recall_95ci'][1]:.3f}]"} for n, c in CI.items()}).T
display(ci_tbl.round(3))
print(f"Intervals come from resampling {len(split_motors['test'])} test motors {B} times. Wide intervals mean the data cannot separate models that look different.")

## 17. Confusion matrices and detailed metrics
**How to read a confusion matrix:** rows = actual condition, columns = predicted. The row-normalised version shows **recall** on its diagonal (of all real `Danger` records, what fraction were found?). The cell *actual Danger → predicted Normal* is the most serious error for maintenance, because a developing fault goes unnoticed.

In [ ]:
for name in ["ANN", "RandomForest", "LogisticRegression"]:
    fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
    plot_cm(ax[0], TEST_METRICS[name]["confusion_matrix"], f"{name}: counts", False)
    plot_cm(ax[1], TEST_METRICS[name]["confusion_matrix"], f"{name}: row-normalised (recall)", True)
    plt.tight_layout(); plt.show()

In [ ]:
for name in ["ANN", "RandomForest", "LogisticRegression"]:
    print(f"\n=== {name}: per-class results on the TEST set ===")
    display(pd.DataFrame(TEST_METRICS[name]["per_class"]).T.round(3))

extra = pd.DataFrame({n: {"accuracy": m["accuracy"], "balanced_accuracy": m["balanced_accuracy"], "macro_F1": m["macro_f1"],
                          "MCC": m["mcc"], "quadratic_weighted_kappa": m["weighted_kappa_quadratic"],
                          "ROC_AUC_ovr_macro": m["roc_auc_ovr_macro"], "PR_AUC_macro": m["pr_auc_macro"], "log_loss": m["log_loss"],
                          "abnormal_recall (Alert+Danger)": m["abnormal_recall"], "abnormal_precision": m["abnormal_precision"],
                          "Danger_predicted_as_Normal (count)": m["danger_predicted_as_normal"]} for n, m in TEST_METRICS.items()}).T
display(extra.round(3))
print("MCC and kappa correct for chance agreement. Quadratic kappa punishes Danger->Normal more than Danger->Alert (the classes are ordered).")
print("PR-AUC is more informative than ROC-AUC when a class is rare.")

pm = pd.DataFrame({"motor_id": g_test, "correct": test_proba["ANN"].argmax(1) == y_test, "danger": y_test == 2})
per_motor = pm.groupby("motor_id").agg(records=("correct", "size"), accuracy=("correct", "mean"), danger_records=("danger", "sum")).round(3)
print("\nANN accuracy per TEST motor (shows how much performance varies between motors):"); display(per_motor)

## 18. Class imbalance and its effect on maintenance decisions

**Why imbalance matters.** `Normal` is the large majority and `Danger` the small minority. A model trained to minimise average error can do well by predicting `Normal` nearly always. In maintenance, the cost of the two mistakes is very different:

| Mistake | Consequence |
|---|---|
| **False alarm** (Normal → predicted Alert/Danger) | An unnecessary inspection: cost of labour and time |
| **Missed fault** (Danger → predicted Normal) | A developing defect is ignored: possible breakdown, secondary damage, safety risk |

This is why **per-class recall for Danger** and **macro-F1** are reported next to accuracy.

**How this notebook handles imbalance (and what it does not do):**
* **Class weights** (training data only): errors on rare classes count more in the loss. They add no artificial rows.
* **No oversampling (e.g. SMOTE)**: synthetic rows would be interpolations between readings, and with repeated readings per motor they would create near-duplicates, so they risk exactly the leakage the split avoids. If you add it later, apply it to the training split only, **after** splitting.
* **Decision threshold:** the model outputs probabilities. Which probability should trigger an alert is a **business decision** that needs real costs. It is not tuned here.

The next cells compute, from your own validation runs, what class weighting did.

In [ ]:
tr = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("_")} for r in ann_trials])
tr["class_weight"] = tr["class_weight"].fillna("none").astype(str)
imb_effect = tr.groupby("class_weight")[["val_macro_F1", "val_Danger_recall"]].agg(["mean", "max"]).round(3)
print("ANN validation results grouped by whether class weighting was used (all search trials):")
display(imb_effect)
print("Compare 'none' vs 'balanced': weighting usually trades some precision for higher recall on rare classes. Read your own numbers; do not assume.")

# Optional: expected cost. The cost values are YOURS to supply from the factory; none are invented here.
COST_MATRIX = None   # example format (rows = actual, cols = predicted):  [[0, 1, 1], [5, 0, 1], [50, 10, 0]]   <- illustration of the SHAPE only
if COST_MATRIX is not None:
    cm_ = np.array(TEST_METRICS["ANN"]["confusion_matrix"]); cost = (cm_ * np.array(COST_MATRIX)).sum() / cm_.sum()
    print(f"Average cost per record on test (ANN) with your cost matrix: {cost:.3f}")
else:
    print("\nCOST_MATRIX is None: expected maintenance cost is not computed, because real cost figures have not been provided.")

### 18b. What is still required for Option B (genuine future failure prediction)
The current file **cannot** support the claim *"the model predicts failures before they happen"*. To do Option B you need:

1. **Verified failure events:** for each motor, the date/time of real failures or fault-driven repairs, with a source (work orders, maintenance logs). `Danger` ≠ failure.
2. **A written failure definition:** what counts as a failure (bearing replaced? unplanned stop? winding fault?).
3. **A prediction horizon H**, chosen with an engineer (for example 14 days): the target becomes *"does a failure occur in the next H days after this reading?"*
4. **Censoring rules:** readings after a repair belong to a new life cycle, and readings too close to the end of the log cannot be labelled.
5. **A chronological (blocked) split per motor**, because the question is about the future, plus a motor-grouped check.
6. **Different metrics:** precision-recall AUC, **lead time** (how early the warning came), and **false alarms per motor-month**.
7. **Enough events.** A handful of failures cannot support a trained model; report the count honestly.

Until these exist, the correct claim is: *"The ANN classifies recorded condition labels for unseen motors."* Nothing more.

In [ ]:
event_like = [c for c in df.columns if any(w in c.lower() for w in ("fail", "event", "repair", "maint", "breakdown"))]
print("Columns that look like failure/repair events:", event_like or "none")
OPTION_B_FEASIBLE = False
print("Option B feasible with the current file:", OPTION_B_FEASIBLE, "(no verified failure-event timestamps)")

## 19. Save the final model and preprocessing artifacts (versioned bundle)

### What goes in the bundle and why
| File | Contents | Safety note |
|---|---|---|
| `model.keras` | Full Keras model (for retraining, inspection, or TensorFlow serving) | Keras native format, no pickle |
| `weights.npz` + `model_architecture.json` | The same network as plain NumPy arrays | Loaded with `allow_pickle=False`: **cannot execute code** |
| `preprocessing.json` | Feature order, scaler mean and scale (fitted on training motors only) | Plain JSON |
| `feature_schema.json` | Names, units, descriptions, observed training min/max | Observed ranges are informational, **not** enforced limits |
| `class_mapping.json` | `0 = Normal, 1 = Alert, 2 = Danger` | Plain JSON |
| `training_metadata.json` | Version, seeds, settings, library versions, data fingerprint, validation/test metrics, provenance flag | Plain JSON |
| `parity_probe.json` | A few artificial inputs with the probabilities this notebook produced | For backend regression tests |
| `reference_inference.py` | NumPy-only loader and predictor (copy it into the backend) | ~80 lines, reviewable |
| `model_card.md`, `reproducibility_report.md` | Documentation | |
| `MANIFEST.json` | SHA-256 of every file above | Detects corruption or tampering |

**Why not pickle/joblib?** Pickle files can run arbitrary code when loaded, which is risky for a web server. A small MLP is only matrix multiplications, so the backend can run it with NumPy alone, with no TensorFlow installed, a much smaller deployment, and numerically identical results (verified below).

### How the FastAPI backend should load the bundle
1. Put the unzipped folder in `ml/artifacts/<bundle_name>/` (git-ignored) and point a setting such as `MODEL_BUNDLE_DIR` at it.
2. At startup, call `load_bundle(dir)` once. It **verifies every SHA-256** in `MANIFEST.json` first and refuses to load a modified bundle.
3. On each request: validate the 14 readings (all present, no unknown names, finite numbers), then call `predict(bundle, readings)`. It standardises in the saved order, runs the network and returns the three class probabilities.
4. Expose `bundle_version`, `label_provenance_documented` and the `PROVISIONAL` flag in the API response and show them in the UI.
5. Add a test that feeds `parity_probe.json` through the backend and checks the same probabilities (tolerance 1e-5).

The Stage-1 backend loads a *binary scikit-learn* model, so it needs a small new loader for this multi-class bundle. That is a separate, planned step.

In [ ]:
import shutil
VERSION_TAG = f"v{VERSION}"
BUNDLE_NAME = f"pm_condition_ann_{VERSION_TAG}"
BUNDLE = OUTPUT_ROOT / BUNDLE_NAME
if BUNDLE.exists(): shutil.rmtree(BUNDLE)
BUNDLE.mkdir(parents=True)

def write_json(name, obj):
    (BUNDLE / name).write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")

# 1. full Keras model
ann_final.save(BUNDLE / "model.keras")

# 2. the same network as plain arrays (no pickle)
dense_layers = [l for l in ann_final.layers if isinstance(l, keras.layers.Dense)]
arrays, arch_layers = {}, []
for i, l in enumerate(dense_layers):
    W, b = l.get_weights(); arrays[f"W{i}"] = W.astype("float64"); arrays[f"b{i}"] = b.astype("float64")
    arch_layers.append({"type": "dense", "units": int(l.units), "activation": l.activation.__name__})
assert [a["activation"] for a in arch_layers] == ["relu"] * (len(arch_layers) - 1) + ["softmax"]
np.savez(BUNDLE / "weights.npz", **arrays)
write_json("model_architecture.json", {"layers": arch_layers, "input_features": len(FEATURES),
                                       "note": "Dropout and L2 are training-time only and are inactive at inference."})

# 3. preprocessing + schema + classes
write_json("preprocessing.json", {"type": "StandardScaler", "formula": "(x - mean) / scale", "feature_names": FEATURES,
                                  "mean": scaler.mean_.tolist(), "scale": scaler.scale_.tolist(),
                                  "fitted_on": "training motors only", "n_fit_samples": int(scaler.n_samples_seen_),
                                  "missing_value_policy": "none: inputs must be complete and finite (no imputation)"})
write_json("feature_schema.json", {"order_is_contract": True, "features": [
    {"name": f, "unit": FEATURE_META[f]["unit"], "description": FEATURE_META[f]["description"],
     "observed_train_min": float(X_train[:, i].min()), "observed_train_max": float(X_train[:, i].max())} for i, f in enumerate(FEATURES)],
    "note": "observed_train_min/max describe the training data. They are NOT validity limits or alarm thresholds."})
write_json("class_mapping.json", {"ordered_by_severity": True, "classes": [{"id": i, "name": n} for i, n in enumerate(CLASS_NAMES)]})

# 4. artificial probe inputs for backend regression tests (NOT real data)
mu, sd = scaler.mean_, scaler.scale_
probes = {"training_mean": mu, "mean_plus_1sd": mu + sd, "mean_minus_1sd": mu - sd}
probe_out = []
for name, vec in probes.items():
    pr = ann_final.predict(((vec - mu) / sd).reshape(1, -1), verbose=0)[0]
    probe_out.append({"name": name, "readings": {f: float(v) for f, v in zip(FEATURES, vec)}, "expected_probabilities": {c: float(p) for c, p in zip(CLASS_NAMES, pr)}})
write_json("parity_probe.json", {"tolerance": 1e-5, "note": "Artificial inputs built from training mean and SD. For software parity tests only.", "probes": probe_out})

In [ ]:
REFERENCE_CODE = r"""# reference_inference.py: NumPy-only inference for the motor-condition ANN bundle.
# Copy this file into the backend. It needs only numpy (no TensorFlow, no pickle).
import hashlib
import json
from pathlib import Path

import numpy as np


def verify_manifest(bundle_dir):
    d = Path(bundle_dir)
    manifest = json.loads((d / "MANIFEST.json").read_text(encoding="utf-8"))
    for name, expected in manifest["files"].items():
        if hashlib.sha256((d / name).read_bytes()).hexdigest() != expected:
            raise ValueError("Bundle file is missing, changed or corrupted: " + name)
    return manifest


def load_bundle(bundle_dir, verify=True):
    d = Path(bundle_dir)
    if verify:
        verify_manifest(d)
    pre = json.loads((d / "preprocessing.json").read_text(encoding="utf-8"))
    arch = json.loads((d / "model_architecture.json").read_text(encoding="utf-8"))
    classes = json.loads((d / "class_mapping.json").read_text(encoding="utf-8"))["classes"]
    meta = json.loads((d / "training_metadata.json").read_text(encoding="utf-8"))
    with np.load(d / "weights.npz", allow_pickle=False) as z:
        layers = [(z["W%d" % i].copy(), z["b%d" % i].copy(), spec["activation"]) for i, spec in enumerate(arch["layers"])]
    return {"feature_names": pre["feature_names"], "mean": np.array(pre["mean"], dtype="float64"),
            "scale": np.array(pre["scale"], dtype="float64"), "layers": layers,
            "class_names": [c["name"] for c in sorted(classes, key=lambda c: c["id"])],
            "version": meta["bundle_version"], "provisional": meta["provisional_label_provenance"]}


def predict_proba_matrix(bundle, X):
    z = (np.asarray(X, dtype="float64") - bundle["mean"]) / bundle["scale"]
    for W, b, act in bundle["layers"]:
        z = z @ W + b
        if act == "relu":
            z = np.maximum(z, 0.0)
        elif act == "softmax":
            z = np.exp(z - z.max(axis=1, keepdims=True))
            z = z / z.sum(axis=1, keepdims=True)
        else:
            raise ValueError("Unsupported activation: " + act)
    return z


def predict(bundle, readings):
    names = bundle["feature_names"]
    missing = [n for n in names if n not in readings]
    unknown = [k for k in readings if k not in names]
    if missing or unknown:
        raise ValueError("missing=%s unknown=%s" % (missing, unknown))
    x = np.array([[float(readings[n]) for n in names]], dtype="float64")
    if not np.isfinite(x).all():
        raise ValueError("All readings must be finite numbers.")
    p = predict_proba_matrix(bundle, x)[0]
    k = int(p.argmax())
    return {"predicted_class": bundle["class_names"][k], "predicted_class_id": k,
            "probabilities": {c: float(v) for c, v in zip(bundle["class_names"], p)},
            "bundle_version": bundle["version"], "provisional_label_provenance": bundle["provisional"]}
"""
(BUNDLE / "reference_inference.py").write_text(REFERENCE_CODE, encoding="utf-8")
print("reference_inference.py written (", len(REFERENCE_CODE.splitlines()), "lines )")

### 19b. Inference-parity test
A deployed model is only trustworthy if it gives **the same answers as the notebook**. Here the NumPy reference code (what the backend will run) is compared with the Keras model on the **raw, unscaled validation readings**, so preprocessing is exercised end to end. Validation data is used (not test), because this checks the software, not performance.

In [ ]:
import importlib.util
# training_metadata.json is needed by load_bundle; write a provisional first version now, completed in Section 20.
write_json("training_metadata.json", {"bundle_version": VERSION_TAG, "provisional_label_provenance": PROVISIONAL})
spec = importlib.util.spec_from_file_location("reference_inference", BUNDLE / "reference_inference.py")
ref = importlib.util.module_from_spec(spec); spec.loader.exec_module(ref)
bundle_loaded = ref.load_bundle(BUNDLE, verify=False)

keras_p = ann_final.predict(X_val_s, verbose=0)
numpy_p = ref.predict_proba_matrix(bundle_loaded, X_val)               # RAW inputs; scaling happens inside
max_diff = float(np.abs(keras_p - numpy_p).max())
same_class = bool((keras_p.argmax(1) == numpy_p.argmax(1)).all())
one = ref.predict(bundle_loaded, dict(zip(FEATURES, X_val[0])))
PARITY = {"validation_rows_compared": int(len(X_val)), "max_abs_probability_difference": max_diff, "identical_predicted_classes": same_class}
print(PARITY); print("single-record example:", one)
assert max_diff < 1e-5 and same_class, "Parity FAILED: do not deploy this bundle."
for p_ in probe_out:
    got = ref.predict(bundle_loaded, p_["readings"])["probabilities"]
    assert all(abs(got[c] - p_["expected_probabilities"][c]) < 1e-5 for c in CLASS_NAMES), "probe mismatch"
print("Parity test PASSED: NumPy reference inference matches Keras (also on the 3 probe inputs).")

## 20. Model card and reproducibility report
Generated from the **actual results of this run**, so the documents cannot drift from the numbers. Read both before you rely on them: the "Limitations" section is deliberately blunt.

In [ ]:
def fmt(v): return "n/a" if v is None else (f"{v:.3f}" if isinstance(v, float) else str(v))
def md_table(d, index_name=""):
    cols = [index_name] + [str(c) for c in d.columns]
    out = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for idx, row in d.iterrows(): out.append("| " + " | ".join([str(idx)] + [fmt(x) for x in row.values]) + " |")
    return "\n".join(out)

now = dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")
env = {"python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__, "scikit-learn": sklearn.__version__,
       "tensorflow": tf.__version__, "keras": keras.__version__, "platform": platform.platform(),
       "gpu": [d.name for d in tf.config.list_physical_devices("GPU")] or "none"}
ann_test = TEST_METRICS["ANN"]
split_summary = {s: {"motors": len(split_motors[s]), "records": int((df.split == s).sum())} for s in split_motors}

training_metadata = {
    "bundle_version": VERSION_TAG, "created_utc": now, "task": "3-class motor condition classification (Option A)",
    "provisional_label_provenance": PROVISIONAL, "label_provenance": LABEL_PROVENANCE,
    "deployable_for_decisions": False, "deployment_note": "Research prototype. Decision support only; not a safety system.",
    "option_b_future_failure_prediction_supported": OPTION_B_FEASIBLE,
    "seed": SEED, "fast_debug_run": FAST_DEBUG,
    "dataset": {"file_name": load_report["file_name"], "sha256": load_report["sha256"], "records": int(len(df)), "motors": int(df.motor_id.nunique()),
                "time_range": [str(df.measured_at.min()), str(df.measured_at.max())], "timestamp_format_assumed": TIMESTAMP_FORMAT},
    "split": {"strategy": "motor-grouped, class-mix balanced by seeded search over label counts only", "fractions": SPLIT_FRACTIONS,
              "min_class_records_per_split": MIN_CLASS_RECORDS, "summary": split_summary},
    "features": FEATURES, "excluded_columns": EXCLUDED, "classes": CLASS_NAMES,
    "ann_config": ANN_CFG, "ann_best_epoch": best_ann["best_epoch"], "class_weights_used": CLASS_WEIGHT_BALANCED if ANN_CFG["class_weight"] == "balanced" else None,
    "baseline_params": {"logistic_regression": LR_PARAMS, "random_forest": RF_PARAMS},
    "validation_summary": VAL_SUMMARY.reset_index().to_dict("records"), "seed_stability_val_macro_f1": SEED_STABILITY,
    "test_metrics": TEST_METRICS, "test_bootstrap_ci_by_motor": CI, "test_evaluated_at_utc": TEST_EVALUATED_AT,
    "label_rule_probe_val": LABEL_PROBE, "leakage_demo_train_val_only": LEAKAGE_DEMO,
    "parity_test": PARITY, "environment": env,
}
write_json("training_metadata.json", training_metadata)

assumptions = [
    "'Header' and 'Vrtex' (as typed in the file) are assumed to mean horizontal and vertical. Unconfirmed.",
    "The type of velocity behind 'mm/s' (RMS, peak, ...) is not specified in the file.",
    "The definition and measurement band of 'gE' are not specified in the file.",
    "Whether rotational speed and load are measured or nominal is unconfirmed.",
    f"Timestamps are read month-first ({TIMESTAMP_FORMAT}); inferred from the data, not confirmed by the data owner.",
]
limitations = [
    f"**Label provenance:** {'NOT documented. All results are provisional.' if PROVISIONAL else 'documented (see table).'} Metrics measure agreement with these labels, not with true machine health.",
    "**Not failure prediction:** the file has no verified failure-event timestamps; `Danger` is a condition state, not a failure.",
    f"**Few motors:** the test set holds {len(split_motors['test'])} motors. Scores are uncertain (see confidence intervals) and may change materially with another motor split.",
    "**Single dataset:** no evidence about other plants, motor types, sensors or mounting conditions.",
    "**Possible label-rule learning:** if labels were derived from sensor thresholds, the network may reproduce the rule (see Section 12b diagnostic).",
    "**No calibration or cost analysis:** probabilities were not calibrated and no real maintenance costs were available.",
    "**Single time-point input:** the model sees one reading, not the history of the motor.",
]
SMOKE_BANNER = "> **SMOKE-TEST RUN (FAST_DEBUG): these results are INVALID and must not be reported or deployed.**\n" if FAST_DEBUG else ""
card = f"""# Model card: Motor condition classifier (ANN) {VERSION_TAG}

> **{STAMP}**
{SMOKE_BANNER}> Research prototype for a final-year project. Decision support only. Not a safety system.

## Model details
- Task: classify motor condition as Normal / Alert / Danger from 14 inputs (Option A).
- Architecture: MLP, {ANN_CFG['hidden_layers']} hidden layer(s) x {ANN_CFG['units']} units (ReLU), dropout {ANN_CFG['dropout']}, L2 {ANN_CFG['l2']}, softmax output; Adam lr {ANN_CFG['lr']}; class weighting: {ANN_CFG['class_weight']}; best epoch {best_ann['best_epoch']}.
- Created: {now} | Seed: {SEED} | Frameworks: TensorFlow {tf.__version__}, Keras {keras.__version__}, scikit-learn {sklearn.__version__}.

## Intended use
- Exploring whether recorded motor condition can be classified from sensor readings for **unseen motors**, as a research artefact.
## Not intended for
- Safety decisions, replacing engineering inspection, or predicting **future failures** (no verified failure events were used).

## Data
- File `{load_report['file_name']}` (SHA-256 `{load_report['sha256'][:16]}...`), {len(df)} records, {df.motor_id.nunique()} motors, {df.measured_at.min():%Y-%m-%d} to {df.measured_at.max():%Y-%m-%d}.
- Class counts: {', '.join(f'{c} {int(n)}' for c, n in cls_counts.items())}.
- Inputs (in this order): {', '.join(FEATURES)}.
- Excluded on purpose: {', '.join(EXCLUDED)}.

## Label provenance
| Question | Answer |
|---|---|
| Documented? | {LABEL_PROVENANCE['documented']} |
| Assigned by | {LABEL_PROVENANCE['assigned_by']} |
| Rule or standard | {LABEL_PROVENANCE['rule_or_standard']} |
| Assigned from the same sensor columns? | {LABEL_PROVENANCE['assigned_from_same_sensor_columns']} |
| Evidence | {LABEL_PROVENANCE['evidence_reference']} |

## Evaluation design
- Motor-grouped split: train {split_summary['train']['motors']} motors / {split_summary['train']['records']} records, validation {split_summary['val']['motors']} / {split_summary['val']['records']}, test {split_summary['test']['motors']} / {split_summary['test']['records']}.
- Standardisation fitted on training motors only. Tuning used validation data only. Test set evaluated once.

## Results
**Validation (used for tuning, optimistic):**

{md_table(VAL_SUMMARY, 'model')}

**Test (single evaluation, {len(y_test)} records, {len(split_motors['test'])} motors):**

{md_table(test_table, 'model')}

**Test uncertainty (95% bootstrap intervals, resampling motors):**

{md_table(ci_tbl, 'model')}

**ANN per-class test results:**

{md_table(pd.DataFrame(ann_test['per_class']).T, 'class')}

- Confusion matrix (rows actual, columns predicted; Normal, Alert, Danger): {ann_test['confusion_matrix']}
- Danger records predicted as Normal: {ann_test['danger_predicted_as_normal']}
- Seed stability (validation macro-F1, mean +/- sd over {len(SEED_STABILITY['seeds'])} seeds): {SEED_STABILITY['mean']:.3f} +/- {SEED_STABILITY['std']:.3f}
- These numbers come from this run on this dataset. They do not demonstrate real-world failure-prediction ability.

## Assumptions (unconfirmed)
""" + "\n".join(f"- {a}" for a in assumptions) + "\n\n## Limitations\n" + "\n".join(f"- {l}" for l in limitations) + f"""

## Loading
Use `reference_inference.py` (NumPy only): `bundle = load_bundle(dir)` then `predict(bundle, readings)`. The loader verifies `MANIFEST.json` checksums first.
"""
(BUNDLE / "model_card.md").write_text(card, encoding="utf-8")
print(card[:1500], "\n... (full card saved in the bundle)")

In [ ]:
repro = f"""# Reproducibility report: {BUNDLE_NAME}
Generated {now}. Run mode: {'FAST_DEBUG (smoke test only, NOT valid for reporting)' if FAST_DEBUG else 'full'}.

## How to reproduce
1. Same notebook version, same input file (verify SHA-256 below), same library versions.
2. Runtime -> Restart session, then Run all. All seeds are fixed (SEED = {SEED}); TensorFlow deterministic ops are enabled.
3. Exact bit-for-bit equality is expected on the same hardware/libraries; small differences can appear on different CPUs/GPUs or library versions.

## Input data
- File: {load_report['file_name']} | SHA-256: `{load_report['sha256']}` | bytes: {load_report['bytes']}
- Raw shape (no header assumed): {load_report['raw_shape']} -> {load_report['data_rows']} data rows after removing the sub-heading row (removed only after an exact 15-cell match).
- The original file was only read, never modified.

## Environment
{json.dumps(env, indent=2)}

## Settings
- Split fractions {SPLIT_FRACTIONS} (by motors), min class records per split {MIN_CLASS_RECORDS}, {N_SPLIT_TRIALS} candidate partitions, {n_feasible} feasible.
- ANN search: {N_ANN_TRIALS} random configurations from {json.dumps(ANN_SPACE)}; early stopping patience {PATIENCE}, max epochs {MAX_EPOCHS}.
- Selected ANN: {json.dumps(ANN_CFG)} | Logistic Regression: {json.dumps(LR_PARAMS)} | Random Forest: {json.dumps(RF_PARAMS)}
- Selection metric: validation macro-F1 (ties: validation Danger recall). Bootstrap resamples: {B}.

## Motors per split (identifiers, treat as internal)
{json.dumps(split_motors, indent=1)}

## Integrity checks that passed in this run
- Unique record IDs; exact sub-heading match; strict numeric and timestamp conversion; labels in the allowed set.
- No excluded column among inputs; no motor shared between splits; scaler fitted on {int(scaler.n_samples_seen_)} training rows only.
- NumPy reference inference matches Keras: max |difference| = {PARITY['max_abs_probability_difference']:.2e}.
- Test set evaluated once, at {TEST_EVALUATED_AT}.
"""
(BUNDLE / "reproducibility_report.md").write_text(repro, encoding="utf-8")

# MANIFEST last: checksums of every other file
files = sorted(p.name for p in BUNDLE.iterdir() if p.is_file() and p.name != "MANIFEST.json")
manifest = {"bundle": BUNDLE_NAME, "files": {n: hashlib.sha256((BUNDLE / n).read_bytes()).hexdigest() for n in files}}
write_json("MANIFEST.json", manifest)

# final proof: reload with checksum verification
bundle_verified = ref.load_bundle(BUNDLE, verify=True)
assert np.abs(ref.predict_proba_matrix(bundle_verified, X_val) - keras_p).max() < 1e-5
print("Bundle written and re-verified from disk:", BUNDLE)
for n in files + ["MANIFEST.json"]: print(f"  {n:30s} {(BUNDLE / n).stat().st_size / 1024:8.1f} KB")

# cleaned working copy (confidential: keep in Colab/Drive, never on GitHub)
df.drop(columns=["split", "y"]).to_csv(OUTPUT_ROOT / "motor_clean_working_copy.csv", index=False)
zip_path = shutil.make_archive(str(OUTPUT_ROOT / BUNDLE_NAME), "zip", OUTPUT_ROOT, BUNDLE_NAME)
print("\nZip:", zip_path)
try:
    from google.colab import files as colab_files
    colab_files.download(zip_path)
except ImportError:
    print("(Not running in Colab: copy the zip from the path above.)")

## Viva defence notes (read these aloud before your viva)

**1. "Why a motor-grouped split?"** Readings from one motor are near-duplicates, so a random row split can let the model "recognise the motor". Section 11b measures this on your own data: it scores the same model with a row split and a grouped split. **Report whatever it shows.** If the row-split score is clearly higher, the gap is leakage. If the two are similar, say that too; the grouped split is still the correct design for the question "does it work on unseen motors?".

**2. "Why macro-F1 and recall, not accuracy?"** Normal dominates. A model that always says Normal gets high accuracy and detects no faults (the Dummy baseline proves it). Macro-F1 weights the three classes equally; Danger recall is the safety-relevant number.

**3. "Why fit the scaler on training data only?"** Fitting on all data lets test information influence training: a form of leakage. The same saved mean/scale are applied at inference.

**4. "How do you know you did not tune on the test set?"** Tuning ranked candidates by validation macro-F1 only. The test set is evaluated once by a guarded function, after all models were frozen. The ANN was declared the primary model beforehand.

**5. "Your results are high. Are they real?"** They are real *for this label on this data*. Section 12b checks whether a tiny rule on velocity reproduces the labels. If the labels came from such a rule, the ANN mostly rediscovered it. Provenance (Section 2) decides how to interpret the numbers.

**6. "Is this failure prediction?"** No. `Danger` is a condition state. Future-failure prediction needs verified failure timestamps and a stated prediction horizon (Section 18b). This work is a condition-classification study and a foundation for that.

**7. "Why class weights and not SMOTE?"** Weights add no synthetic rows. SMOTE interpolates between readings, which with repeated per-motor readings can create near-duplicates and leakage. If used, it must touch the training split only, after splitting.

**8. "How reliable is the test score?"** Check the bootstrap intervals: only a handful of test motors exist, so intervals are wide. Say so plainly; do not claim one model beats another unless the intervals support it.

**9. "Why can the backend run the model without TensorFlow?"** The ANN is matrix multiplications and ReLU/softmax. Weights are stored as NumPy arrays (no pickle), and a parity test proves identical outputs to Keras.

**10. "What would you do next?"** Document label provenance; obtain failure events; define a horizon; use chronological splits per motor; add real maintenance costs to set the alert threshold; test on motors from another site.